# LikhKeDo · free Evon server (Kaggle GPU T4 x2)

Runs **Evon v3.3 · community 4-bit GGUF (llama.cpp)** — `luminenio/gnani-evon-v3.3-30B-A3B-GGUF`, Q4_K_M, made from Gnani's BF16 weights — behind a Cloudflare quick tunnel. See `EVON_FREE.md` in the repo.

Before running:
1. Settings → Accelerator: **GPU T4 x2**, Internet: **on** (needs a phone-verified Kaggle account).
2. Add-ons → Secrets: `HF_TOKEN` (a Hugging Face token whose account was granted access to the GGUF repo) and `EVON_API_KEY` (any long random string; put the same value in your local `.env`).
3. Run all cells. The last cell prints the tunnel URL. **Stop the session when you are done** (weekly GPU quota).

Limits: the tunnel URL changes every session, and Cloudflare cuts any single request after 100 s (LikhKeDo uses `EVON_TIMEOUT=90` and one batched verification call).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from kaggle_secrets import UserSecretsClient
import os, subprocess, time, urllib.request, json, re
_s = UserSecretsClient()
os.environ["HF_TOKEN"] = _s.get_secret("HF_TOKEN")
KEY = _s.get_secret("EVON_API_KEY")  # never printed
print("secrets loaded")

In [ ]:
# Build llama-server with CUDA for the T4 (compute capability 7.5). Takes ~10 minutes.
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp
!cmake -S /kaggle/working/llama.cpp -B /kaggle/working/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF > /kaggle/working/cmake.log 2>&1; tail -3 /kaggle/working/cmake.log
!cmake --build /kaggle/working/llama.cpp/build --config Release -j 4 --target llama-server 2>&1 | tail -3
!ls -la /kaggle/working/llama.cpp/build/bin/llama-server

In [ ]:
# Download the 4-bit GGUF (~24.7 GB) plus its licence files.
!pip -q install -U huggingface_hub
!hf download luminenio/gnani-evon-v3.3-30B-A3B-GGUF gnani-evon-v3.3-30B-A3B-Q4_K_M.gguf LICENSE LICENSE-NVIDIA NOTICE --local-dir /kaggle/working/models
!ls -la /kaggle/working/models

In [ ]:
# Start llama-server: OpenAI-compatible, model alias "evon", split across both T4s, one request at a time.
cmd = ("/kaggle/working/llama.cpp/build/bin/llama-server "
       "-m /kaggle/working/models/gnani-evon-v3.3-30B-A3B-Q4_K_M.gguf --alias evon "
       "-ngl 99 --split-mode layer --tensor-split 1,1 -c 8192 -np 1 --jinja "
       f"--host 127.0.0.1 --port 8000 --api-key {KEY}")
server = subprocess.Popen(cmd + " > /kaggle/working/server.log 2>&1", shell=True)
for i in range(120):  # loading 24.7 GB onto the GPUs takes a few minutes
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5) as r:
            if r.status == 200:
                print(f"server ready after {i * 5} s"); break
    except Exception:
        pass
    time.sleep(5)
else:
    print("server not ready; see the log below")
!tail -5 /kaggle/working/server.log

In [ ]:
# Local test request (thinking off, JSON only), with timing.
body = {"model": "evon", "temperature": 0.6, "top_p": 0.95, "max_tokens": 200,
        "chat_template_kwargs": {"enable_thinking": False},
        "messages": [{"role": "user", "content": "Reply in Hindi in Devanagari script, one sentence: what is a free-look period? Return ONLY JSON {\"text\": \"...\"}"}]}
req = urllib.request.Request("http://127.0.0.1:8000/v1/chat/completions", data=json.dumps(body).encode(),
                             headers={"Content-Type": "application/json", "Authorization": f"Bearer {KEY}"})
t0 = time.time()
with urllib.request.urlopen(req, timeout=120) as r:
    out = json.load(r)
print(f"{time.time() - t0:.1f} s", out["choices"][0]["message"]["content"], out.get("usage"))

In [ ]:
# Cloudflare quick tunnel (no account). The URL changes every session.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared && chmod +x /kaggle/working/cloudflared
tunnel = subprocess.Popen("/kaggle/working/cloudflared tunnel --url http://127.0.0.1:8000 > /kaggle/working/tunnel.log 2>&1", shell=True)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/kaggle/working/tunnel.log").read())
    if m:
        url = m.group(0); break
print("Put these in your local .env:")
print(f"EVON_PROVIDER=llamacpp\nEVON_BASE_URL={url}/v1\nEVON_MODEL=evon\nEVON_QUANT=Q4_K_M\nEVON_TIMEOUT=90")
print("EVON_API_KEY=<the same value as the Kaggle secret>")

In [ ]:
# Optional: watch the server while you work (interrupt to stop watching; the server keeps running).
!tail -n 20 -f /kaggle/working/server.log